# TSA_ch12_estimation

**Published in:** Time Series Analysis  
**Author:** Daniel Traian Pele  
**Description:** Comparison of spectral density estimation methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram, welch
from scipy.signal.windows import dpss

COLORS = {'blue':'#1A3A6E','red':'#DC3545','green':'#2E7D32','amber':'#B5853F','orange':'#E6802E','purple':'#8E44AD','gray':'#666666'}

In [ ]:
rng = np.random.default_rng(0)
phi = 0.7
N = 512
eps = rng.normal(0, 1, N + 100)
x = np.zeros(N + 100)
for t in range(1, N + 100):
    x[t] = phi * x[t-1] + eps[t]
x = x[100:]
true_psd = lambda f: 1.0 / (1 - 2*phi*np.cos(2*np.pi*f) + phi**2)

# Raw periodogram
f_raw, p_raw = periodogram(x)
# Welch
f_welch, p_welch = welch(x, nperseg=128)
# Multitaper
tapers, _ = dpss(N, NW=4, Kmax=7, return_ratios=True)
mt_psds = []
for tap in tapers:
    ft = np.fft.rfft(x * tap)
    mt_psds.append(np.abs(ft)**2 / N)
p_mt = np.mean(mt_psds, axis=0)
f_mt = np.fft.rfftfreq(N)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
f_true = np.linspace(0.01, 0.49, 400)

for ax, (fr, ps), lbl, col in zip(axes,
    [(f_raw, p_raw, ), (f_welch, p_welch), (f_mt, p_mt)],
    ['Raw Periodogram', 'Welch Estimator', 'Multitaper'],
    [COLORS['blue'], COLORS['orange'], COLORS['red']]):
    ax.semilogy(fr[1:], ps[1:], color=col, lw=1, alpha=0.8, label=lbl)
    ax.semilogy(f_true, true_psd(f_true), 'k--', lw=1.5, label='True PSD')
    ax.set_title(lbl)
    ax.set_xlabel('Frequency')
    ax.set_ylabel('PSD (log)')
    ax.set_facecolor('none')
    ax.spines[['top','right']].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1, frameon=False)

fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('estimation_comparison.pdf', bbox_inches='tight')
plt.show()